# PI-BDP × N-CMAPSS — 추가본 v1.3: 예측분포 전체를 쓰는 정비 규칙(M1)과 계산 비용(M2)

**실행 조건.** `addendum_v1.3/PROTOCOL_v1.3_addendum.md`와 이 노트북이 GitHub에 커밋되어 있어야 합니다. 그 커밋 주소를 아래 `ADDENDUM_COMMIT_URL`에 넣으세요. 주소가 없으면 실행이 멈춥니다.

**런타임.** CPU(GPU 필요 없음). M1은 몇 분, M2는 약 1시간(길면 1시간 30분) 걸립니다. 끊기면 다시 "모두 실행"하면 됩니다. M2는 끝난 서브셋을 건너뜁니다.

**하는 일**
- **M1:** 저장된 Phase 3 V0 시험 예측과 사전 고정 보정 상수로, 비용비 ρ에 맞춘 비용률 최소화 정비 규칙(ER)을 사후 정비 시뮬레이터에서 평가합니다. 먼저 기존 사후 결과(LB90, TBM, 조정 GB)를 같은 코드로 재현해 확인합니다.
- **M2:** 동결된 PI-BDP와 RF·GB·MLP의 적합·예측 시간을 잽니다(새 결과 수치는 만들지 않음).

**결과.** 드라이브 `addendum_v1.3/`에 저장됩니다. 끝나면 그 폴더의 파일을 올려 주세요.

In [ ]:
import os, sys, json, glob, time, hashlib, pickle, platform, subprocess, importlib
import numpy as np, pandas as pd
from scipy.optimize import minimize
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
TEST_MODE = os.environ.get('NCM_TEST') == '1'   # 합성 자료로 코드만 점검할 때 (실제 실행에서는 쓰지 않음)

ADDENDUM_COMMIT_URL = ''   # <- 추가본 v1.3 커밋 주소를 붙여넣으세요
assert TEST_MODE or ADDENDUM_COMMIT_URL.startswith('https://github.com/'), '먼저 추가본 v1.3 커밋 주소를 ADDENDUM_COMMIT_URL에 넣으세요.'

OUT = f'{ROOT}/addendum_v1.3'; os.makedirs(OUT, exist_ok=True)
cfg = json.load(open(f'{ROOT}/phase1/config.json')); SUBSETS = sorted(cfg['k'])
CAL = {str(k): float(v) for k, v in json.load(open(f'{ROOT}/phase1/calibration.json'))['c'].items()}
C90, EPS = CAL['90'], 0.5
L_LEAD, DT, DP, RHO0 = 5, 1, 2.0, 10
TAU0 = L_LEAD + DT
RHOS = [2, 5, 10, 20, 50]; TAU_GRID = np.arange(0, 41); N_BOOT = 20000
K = ['subset', 'unit', 'cycle']
REF10 = {'PI-BDP q05 (calibrated)': 51.928, 'TBM (Weibull optimal age)': 41.981, 'GB': 47.599}   # 사후 결과 (rho = 10)
print('보정 상수:', CAL, '| commit:', ADDENDUM_COMMIT_URL)

In [ ]:
# === M1 규칙: 비용률 최소화 교체 (추가본 v1.3, 1.2절) ===
QCOLS = ['q025', 'q050', 'q100', 'q250', 'q500', 'q750', 'q900', 'q950', 'q975']
PLEV = np.array([0.025, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.975])
PAIRS = {'95': ('q025', 'q975'), '90': ('q050', 'q950'), '80': ('q100', 'q900'), '50': ('q250', 'q750')}

def quantiles(df, c=None, eps=0.5):
    """9 quantiles per row; with c = {'50','80','90','95' -> constant} the pre-registered conformal scaling
    is applied level by level around the median (lower bounds truncated at 0), then made non-decreasing."""
    Q = df[QCOLS].to_numpy(float).copy()
    if c is not None:
        m = df['q500'].to_numpy(float)
        for lev, (lo, hi) in PAIRS.items():
            i, j = QCOLS.index(lo), QCOLS.index(hi); k = float(c[lev])
            Q[:, i] = np.maximum(m - k * np.maximum(m - Q[:, i], eps), 0)
            Q[:, j] = m + k * np.maximum(Q[:, j] - m, eps)
    Q = np.maximum(Q, 0)
    return np.maximum.accumulate(Q, axis=1)

def knots(q):
    """CDF knots from 9 quantiles: the quantile points (tied values keep the largest level), with
    linear tails at the density of the outermost quantile segments: F falls to 0 at
    q_2.5 - (q_5 - q_2.5) (truncated at RUL = 0, the rest becoming mass at 0) and rises to 1 at
    q_97.5 + (q_97.5 - q_95)."""
    xs, ps = [], []
    for x, p in zip(q, PLEV):
        if xs and x <= xs[-1] + 1e-9: ps[-1] = max(ps[-1], p)
        else: xs.append(float(x)); ps.append(float(p))
    lo_gap = max(q[1] - q[0], 1e-6); hi_gap = max(q[-1] - q[-2], 1e-6)
    x0 = xs[0] - lo_gap * ps[0] / (PLEV[1] - PLEV[0])   # lower tail at the density of the 2.5-5% segment
    if x0 >= 0: xs, ps = [x0] + xs, [0.0] + ps
    else:
        F0 = ps[0] * (-x0) / (xs[0] - x0)          # mass that the linear tail would put below 0
        xs, ps = ([0.0] + xs, [F0] + ps) if xs[0] > 0 else (xs, ps)
    x1 = xs[-1] + hi_gap * (1 - ps[-1]) / (PLEV[-1] - PLEV[-2])
    return np.array(xs + [x1]), np.array(ps + [1.0])

def cdf(q, r):
    """P(RUL <= r): piecewise-linear interpolation of the knots; 0 below and 1 above them."""
    xs, ps = knots(q)
    return np.interp(np.asarray(r, float), xs, ps, left=0.0, right=1.0)

def order_now(q, t, rho, L=5, Dp=2.0):
    """True if replacing L cycles from now minimises the expected long-run downtime rate
    Dp[(1-F(r)) + rho F(r)] / (t + sum_{s<r}(1-F(s))) over r in {L, ..., end of the upper tail}."""
    R = max(L, int(np.ceil(knots(q)[0][-1])))
    r = np.arange(L, R + 1)
    s = np.arange(0, R + 1)
    S = 1 - cdf(q, s)                       # survival P(RUL > s)
    used = t + np.concatenate([[0.0], np.cumsum(S)])[r]   # t + E[min(RUL, r)] = t + sum_{s<r} S(s)
    F = cdf(q, r)
    g = Dp * ((1 - F) + rho * F) / used
    return int(np.argmin(g)) == 0           # ties -> earliest r, i.e. order now

def er_stat(df, c, rho, L=5):
    """Decision statistic for the simulator: 0 where the rule orders maintenance, 1e9 otherwise (threshold 0.5)."""
    Q = quantiles(df, c)
    return np.array([0.0 if order_now(Q[i], int(t), rho, L) else 1e9 for i, t in enumerate(df['cycle'].to_numpy())])

In [ ]:
# === 사후 정비 시뮬레이터 (NCMAPSS_PostHoc_정비시뮬레이션.ipynb와 같은 코드) ===
def cal_lo(df):
    m = df.q500.values
    return np.maximum(m - C90 * np.maximum(m - df.q050.values, EPS), 0)

def frame(df, stat):
    d = df[K].copy(); d['stat'] = stat; d['eol'] = (df.cycle + df.true).values
    return d.sort_values(K).reset_index(drop=True)

def simulate(d, tau):
    out = []
    for (s, u), g in d.groupby(['subset', 'unit'], sort=False):
        eol = int(g.eol.iloc[0]); hit = g.cycle.values[g.stat.values <= tau]
        if len(hit) and hit[0] + L_LEAD < eol: out.append((s, u, eol, hit[0] + L_LEAD, 0))
        else: out.append((s, u, eol, eol, 1))
    return pd.DataFrame(out, columns=['subset', 'unit', 'eol', 'used', 'fail'])

def rate(r, rho):
    dt = np.where(r.fail == 1, rho * DP, DP)
    return 1000 * dt.sum() / r.used.sum()

def summ(r, rho):
    return dict(downtime=rate(r, rho), fail_rate=r.fail.mean(), unused=((r.eol - r.used) / r.eol).mean())

# 시험 예측: V0 (분위수 9개), GB (시드 평균); 개발 예측: GB (임계값 조정용)
v0t = pd.concat([pd.read_csv(f'{ROOT}/phase3/test_PI-BDP_V0_{s}.csv.gz') for s in SUBSETS], ignore_index=True)
v0t = v0t.sort_values(K).reset_index(drop=True)
gbt = pd.read_csv(f'{ROOT}/phase3/test_GB.csv.gz').groupby(K, as_index=False)[['pred', 'true']].mean()
gbd = pd.read_csv(f'{ROOT}/phase2/preds_GB.csv.gz').groupby(K, as_index=False)[['pred', 'true']].mean()
TEST = {'PI-BDP q05 (raw)': frame(v0t, v0t.q050.values), 'PI-BDP q05 (calibrated)': frame(v0t, cal_lo(v0t)),
        'GB': frame(gbt, gbt.pred.values)}
assert TEST['GB'][K].equals(TEST['PI-BDP q05 (raw)'][K]), 'GB와 V0의 시험 시점이 다릅니다'
DEV_GB = frame(gbd, gbd.pred.values)

ref = TEST['PI-BDP q05 (raw)'][K]
units = ref.drop_duplicates(['subset', 'unit'])[['subset', 'unit']].merge(
    TEST['GB'].groupby(['subset', 'unit']).eol.first().reset_index(), on=['subset', 'unit'])
def fixed(fn):
    r = units.copy(); r['used'], r['fail'] = zip(*[fn(e) for e in r.eol]); return r
CM = fixed(lambda e: (e, 1)); ORACLE = fixed(lambda e: (e - 1, 0))

def weib(Ls, cens):
    def nll(p):
        k, lam = np.exp(p); z = Ls / lam
        return -((np.log(k / lam) + (k - 1) * np.log(z) - z ** k)[~cens].sum() - (z[cens] ** k).sum())
    k, lam = np.exp(minimize(nll, [np.log(3), np.log(Ls.mean())], method='Nelder-Mead').x); return k, lam
def tbm_age(k, lam, rho):
    T = np.arange(5, 151); grid = np.linspace(0, 150, 3001); R = np.exp(-(grid / lam) ** k)
    cumR = np.concatenate([[0], np.cumsum((R[1:] + R[:-1]) / 2 * np.diff(grid))])
    RT = np.exp(-(T / lam) ** k); IT = np.interp(T, grid, cumR)
    return int(T[np.argmin((DP * RT + rho * DP * (1 - RT)) / IT)])
WB = {}
for s in SUBSETS:
    d = np.load(f'{ROOT}/phase0/dev_{s}.npz'); an = list(d['A_var'])
    eol = pd.Series(d['A'][:, an.index('cycle')]).groupby(d['A'][:, an.index('unit')]).max().values.astype(float); d.close()
    WB[s] = weib(eol, eol >= 100)
def TBM(rho):
    r = units.copy(); T = r.subset.map(lambda s: tbm_age(*WB[s], rho))
    r['used'] = np.where(T < r.eol, T, r.eol); r['fail'] = (T >= r.eol).astype(int); return r
TUNED_GB = {rho: int(TAU_GRID[int(np.argmin([rate(simulate(DEV_GB, t), rho) for t in TAU_GRID]))]) for rho in RHOS}

# 재현 점검 (실행 조건): 사후 결과와 0.01 이내
chk = {'PI-BDP q05 (calibrated)': rate(simulate(TEST['PI-BDP q05 (calibrated)'], TAU0), RHO0),
       'TBM (Weibull optimal age)': rate(TBM(RHO0), RHO0), 'GB': rate(simulate(TEST['GB'], TUNED_GB[RHO0]), RHO0)}
print('재현 점검 (rho = 10):', {k: round(v, 3) for k, v in chk.items()}, '| 조정 GB tau:', TUNED_GB)
if not TEST_MODE:
    for k, v in chk.items(): assert abs(v - REF10[k]) < 0.01, f'{k}: {v:.3f} != {REF10[k]} -> 멈춤, 원인 확인'
print('시험 엔진', len(units), '대, 사이클', len(v0t))

In [ ]:
# === M1 실행: rho별 ER 결정 -> 시뮬레이션 ===
def er_frame(c, rho):
    return frame(v0t, er_stat(v0t, c, rho, L_LEAD))
ER = {}
for rho in RHOS:
    ER[('ER-cal', rho)] = er_frame(CAL, rho)
    ER[('ER-raw', rho)] = er_frame(None, rho)
def records(policy, rho):
    if policy == 'CM': return CM
    if policy == 'Oracle': return ORACLE
    if policy == 'TBM (Weibull optimal age)': return TBM(rho)
    if policy in ('ER-cal', 'ER-raw'): return simulate(ER[(policy, rho)], 0.5)
    if policy == 'GB tuned': return simulate(TEST['GB'], TUNED_GB[rho])
    return simulate(TEST[policy], TAU0)          # untuned PI-BDP rules (tau = L + dt)
POLICIES = ['CM', 'TBM (Weibull optimal age)', 'Oracle', 'PI-BDP q05 (raw)', 'PI-BDP q05 (calibrated)', 'GB tuned', 'ER-raw', 'ER-cal']
rows = [dict(rho=rho, policy=p, **summ(records(p, rho), rho)) for rho in RHOS for p in POLICIES]
TAB = pd.DataFrame(rows); TAB.to_csv(f'{OUT}/m1_rule_table.csv', index=False)
print(TAB.pivot(index='policy', columns='rho', values='downtime').loc[POLICIES].round(2).to_string())

# 계층 붓스트랩 (rho = 10), 사후 노트북과 같은 표본 생성
rng = np.random.default_rng(1)
subs = sorted(units.subset.unique()); per_unit_idx = {s: np.flatnonzero(units.subset.values == s) for s in subs}
B_idx = []
for _ in range(N_BOOT):
    pick = rng.choice(subs, len(subs))
    B_idx.append(np.concatenate([rng.choice(per_unit_idx[s], len(per_unit_idx[s])) for s in pick]))
def boot_rates(r, rho=RHO0):
    r = r.set_index(['subset', 'unit']).loc[list(zip(units.subset, units.unit))]
    dt = np.where(r.fail.values == 1, rho * DP, DP); used = r.used.values
    return np.array([1000 * dt[i].sum() / used[i].sum() for i in B_idx])
BR = {p: boot_rates(records(p, RHO0)) for p in POLICIES}
ci = []
for p in POLICIES:
    ci.append(dict(policy=p, downtime=rate(records(p, RHO0), RHO0), lo=np.percentile(BR[p], 2.5), hi=np.percentile(BR[p], 97.5)))
CI = pd.DataFrame(ci)
r_lb = CI[CI.policy == 'PI-BDP q05 (calibrated)'].iloc[0]
print(f'붓스트랩 점검: LB90 95% 구간 {r_lb.lo:.3f}–{r_lb.hi:.3f} (사후 결과 29.526–85.890)')
def word(lo, hi): return 'lower downtime' if hi < 0 else ('higher downtime' if lo > 0 else 'no significant difference')
comp = []
for a, b, role in [('ER-cal', 'PI-BDP q05 (calibrated)', 'primary (a)'), ('ER-cal', 'TBM (Weibull optimal age)', 'primary (b)'),
                   ('ER-cal', 'GB tuned', 'primary (c)'), ('ER-raw', 'PI-BDP q05 (calibrated)', 'secondary')]:
    dlt = BR[a] - BR[b]; lo, hi = np.percentile(dlt, [2.5, 97.5])
    comp.append(dict(comparison=f'{a} - {b}', role=role, diff=rate(records(a, RHO0), RHO0) - rate(records(b, RHO0), RHO0), lo=lo, hi=hi, wording=word(lo, hi)))
COMP = pd.DataFrame(comp)
pd.concat([CI.assign(kind='downtime'), COMP.assign(kind='difference')], ignore_index=True).to_csv(f'{OUT}/m1_rho10_ci.csv', index=False)
print(CI.round(3).to_string(index=False)); print(COMP.round(3).to_string(index=False))

per = []
for s in SUBSETS:
    for p in ['PI-BDP q05 (calibrated)', 'ER-cal', 'TBM (Weibull optimal age)', 'GB tuned']:
        r = records(p, RHO0); r = r[r.subset == s]; per.append(dict(subset=s, policy=p, **summ(r, RHO0)))
PER = pd.DataFrame(per); PER.to_csv(f'{OUT}/m1_per_subset.csv', index=False)
print(PER.pivot(index='subset', columns='policy', values='downtime').round(2).to_string())

In [ ]:
# === M2: 계산 비용 (CPU) ===
import shutil, tempfile, gc
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
try:
    cpu = subprocess.run(['bash', '-c', "grep -m1 'model name' /proc/cpuinfo"], capture_output=True, text=True).stdout.strip()
except Exception: cpu = platform.processor()
import scipy, sklearn
ENV = dict(cpu=cpu, n_cpu=os.cpu_count(), python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
           scipy=scipy.__version__, sklearn=sklearn.__version__, gpu_visible=os.path.exists('/dev/nvidia0'))
json.dump(ENV, open(f'{OUT}/m2_env.json', 'w'), indent=1); print(ENV)

LIB, LIB_SHA = f'{ROOT}/freeze_v1/pibdp_ncm.py', '468de08c57f445447e437a9711cdbdf525097333ae883cfef6acfb425b1e1e3c'
assert os.path.exists(LIB), f'{LIB} 가 없습니다 (고정 패키지 노트북이 만든 파일)'
assert sha(LIB) == LIB_SHA, '라이브러리가 동결본과 다릅니다'
LIBDIR = tempfile.mkdtemp(); shutil.copy(LIB, f'{LIBDIR}/pibdp_ncm.py'); sys.path.insert(0, LIBDIR)
import pibdp_ncm as P; importlib.reload(P); assert sha(P.__file__) == LIB_SHA
rec0 = json.load(open(f'{ROOT}/phase0/sha256_phase0.json'))
SEL = json.load(open(f'{ROOT}/phase2/ml_selected.json'))
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
def make(name, p, seed):   # Phase 3와 같음
    p = dict(p)
    if name == 'RF': return RandomForestRegressor(random_state=seed, n_jobs=-1, **p)
    if name == 'GB': return HistGradientBoostingRegressor(random_state=seed, early_stopping=False, **p)
    p['hidden_layer_sizes'] = tuple(p['hidden_layer_sizes'])
    return make_pipeline(StandardScaler(), MLPRegressor(random_state=seed, max_iter=500, **p))

PART = f'{OUT}/m2_parts'; os.makedirs(PART, exist_ok=True)
for s in SUBSETS:
    fpart = f'{PART}/{s}.json'
    if os.path.exists(fpart): print('있음:', s); continue
    ftest = f'{ROOT}/phase0/sealed_test_{s}.npz'
    assert sha(ftest) == rec0[f'sealed_test_{s}.npz'], f'{s}: 시험 파일 해시 불일치'
    Dd = P.load_dev(f'{ROOT}/phase0/dev_{s}.npz'); Dt = P.load_dev(ftest)
    dev_units = list(Dd['units']); test_units = [u for u in Dt['units'] if not Dt['censored'][u]]
    conf = dict(baseline=cfg['baseline'], n_h=cfg['n_h'], k=cfg['k'][s], shrink=cfg['shrink'])
    fit_t = []
    for _ in range(3):
        t0 = time.perf_counter(); m = P.PIBDP(**conf).fit(Dd, dev_units); fit_t.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); m.use_data(Dt); t1 = time.perf_counter()          # 시험 자료 Δz (재적합 없음)
    rows = [r for x in test_units for r in m.predict_unit(x, n=4000)]; t2 = time.perf_counter()
    new = P.score_rows(rows, Dt['eol'])[['unit', 'cycle', 'q050', 'q500']]
    old = v0t.loc[v0t.subset == s, ['unit', 'cycle', 'q050', 'q500']]
    M = old.merge(new, on=['unit', 'cycle'], how='outer', suffixes=('_old', '_new'), indicator=True)
    repro = dict(subset=s, n_stored=len(old), n_new=len(new), n_matched=int((M._merge == 'both').sum()),
                 max_abs_q500=float(np.nanmax(np.abs(M.q500_new - M.q500_old))),
                 max_abs_q050=float(np.nanmax(np.abs(M.q050_new - M.q050_old))))
    tim = [dict(subset=s, model='PI-BDP V0', device='CPU', n_train_units=len(dev_units), n_cycles=len(rows),
                fit_s=float(np.median(fit_t)), fit_s_runs=[round(x, 3) for x in fit_t], test_delta_s=t1 - t0,
                predict_s=t2 - t1, predict_ms_per_cycle=1000 * (t2 - t1) / len(rows))]
    d = pickle.load(open(f'{ROOT}/phase3/deltas_{s}.pkl', 'rb'))
    tr = [u for u in d['dev_units'] if not d['cens_dev'][u]]
    Xtr, ytr, _ = P.fold_xy(d['Zdev'], tr, d['eol_dev'], 10); Xte, yte, _ = P.fold_xy(d['Zte'], d['test_units'], d['eol_te'], 10)
    for name in ['RF', 'GB', 'MLP']:
        t0 = time.perf_counter(); mdl = make(name, SEL[name]['params'], 0).fit(Xtr, ytr); t1 = time.perf_counter()
        mdl.predict(Xte); t2 = time.perf_counter()
        tim.append(dict(subset=s, model=name, device='CPU', n_train_units=len(tr), n_train_rows=len(ytr), n_cycles=len(yte),
                        fit_s=t1 - t0, predict_s=t2 - t1, predict_ms_per_cycle=1000 * (t2 - t1) / len(yte)))
    json.dump(dict(timing=tim, repro=repro), open(fpart, 'w'), indent=1)
    print(s, f"PI-BDP 적합 {np.median(fit_t):.1f} s, 예측 {t2 - t1:.1f} s ({len(rows)} 사이클) | 재현 최대차 q500 {repro['max_abs_q500']:.4f}")
    del Dd, Dt, m, rows, d; gc.collect()

TIM, REPRO = [], []
for s in SUBSETS:
    p = json.load(open(f'{PART}/{s}.json')); TIM += p['timing']; REPRO.append(p['repro'])
TIMING = pd.DataFrame(TIM).drop(columns=['fit_s_runs']); TIMING.to_csv(f'{OUT}/m2_timing.csv', index=False)
REP = pd.DataFrame(REPRO); REP.to_csv(f'{OUT}/m2_repro_check.csv', index=False)
M2SUM = TIMING.groupby('model').agg(fit_s_total=('fit_s', 'sum'), fit_s_max=('fit_s', 'max'), predict_s_total=('predict_s', 'sum'),
                                    n_cycles=('n_cycles', 'sum'))
M2SUM['predict_ms_per_cycle'] = 1000 * M2SUM.predict_s_total / M2SUM.n_cycles
print(M2SUM.round(2).to_string()); print(REP.round(4).to_string(index=False))

In [ ]:
# === 요약 저장 + 해시 ===
txt = '\n'.join([f"Addendum v1.3 (exploratory) {time.strftime('%Y-%m-%d %H:%M:%S')} | commit: {ADDENDUM_COMMIT_URL}",
    f'M1 expected-rate rule: L={L_LEAD}, dt={DT}, Dp={DP}, rhos={RHOS}, calibration constants={CAL}, eps={EPS}',
    'Reproduction check (rho=10): ' + json.dumps({k: round(v, 3) for k, v in chk.items()}) + f' | tuned GB tau={TUNED_GB}',
    '', 'Downtime per 1000 cycles by rho:', TAB.pivot(index='policy', columns='rho', values='downtime').loc[POLICIES].round(3).to_string(),
    '', 'Unplanned failure rate by rho:', TAB.pivot(index='policy', columns='rho', values='fail_rate').loc[POLICIES].round(3).to_string(),
    '', 'Mean unused life by rho:', TAB.pivot(index='policy', columns='rho', values='unused').loc[POLICIES].round(3).to_string(),
    '', 'rho = 10, hierarchical bootstrap 95% CI:', CI.round(3).to_string(index=False), '', COMP.round(3).to_string(index=False),
    '', 'Per subset (rho = 10) downtime:', PER.pivot(index='subset', columns='policy', values='downtime').round(2).to_string(),
    '', 'M2 environment: ' + json.dumps(ENV),
    'M2 timing over subsets (seconds; PI-BDP fit = median of 3 runs per subset; ML = seed 0; deltas precomputed for ML):', M2SUM.round(3).to_string(),
    '', 'M2 reproduction of stored Phase 3 V0 predictions (max absolute difference):', REP.round(5).to_string(index=False)])
open(f'{OUT}/v13_summary.txt', 'w', encoding='utf-8').write(txt)
H = {os.path.relpath(p, OUT): sha(p) for p in sorted(glob.glob(f'{OUT}/**', recursive=True)) if os.path.isfile(p) and 'sha256_v13' not in p}
json.dump(H, open(f'{OUT}/sha256_v13.json', 'w'), indent=1)
print(txt)